# Colour Metrics over Densities - Samples

In [1]:
try:
    import mat73
except ImportError:
    pass

from pathlib import Path
from typing import Sequence

import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import tifffile

In [11]:
path = "../../"
path = Path(path).expanduser()
import sys
sys.path.insert(0, str(path))

In [12]:
import decode
import decode.neuralfitter.inference.functional as infer_func
print(decode.__file__)
log = decode.generic.logging.get_logger(__name__)

%config InlineBackend.figure_format='retina'

/home/shahao/projects/DECODE-Plex/notebook/density/../../decode/__init__.py


In [ ]:
# some paths and hardware settings
path_spline = "../../calibration/FigS5-Pos0_240802_NC_BeadCal_DualColor_Z_1_MMStack_Default.ome_3dcal.mat"
path_trafo = "../../calibration/FigS5-Pos0_240802_NC_BeadCal_DualColor_Z_1_MMStack_Default.ome_trafo.mat" 

path_out = "../../data/density/dual_color"
path_out = Path(path_out).expanduser()

trafo_size_ref = 512
trafo_mirr_dim = 0

device = "cuda:0"

path_spline, path_trafo = Path(path_spline).expanduser(), Path(path_trafo).expanduser()

In [14]:
xextent = (50.5, 205.5)
yextent = xextent
zextent = (-600., 600.)

img_shape = (256, 256)
px_size = (100, 100)  # fake, but doesn't matter
frame_extent = ((-0.5, img_shape[0] - 0.5), (-0.5, img_shape[1] - 0.5))

n_frames = 1000  # per step
em_min = 1e5
frame_min = 50
frame_max = 1000
n_steps = 16

psf_norm = "auto" # because we split by colour, otherwise we would take `colour` of beads

density = torch.logspace(math.log10(0.03), math.log10(3.), n_steps)
area = 127**2 * 1e-6 * (xextent[1] - xextent[0]) * (yextent[1] - yextent[0])  # um^2
n_emitter = density * area

choric_mat = torch.tensor([[0.5, 0.5]])

random_seed = 42  # emitters in the same n_frames / n_steps combination will be the same


In [15]:
# definition of scenarios

# product dimensions
em = pd.DataFrame({"n_emitter": n_emitter.tolist()})
# snr = pd.DataFrame({"snr": ["low", "medium", "high"]})
snr = pd.DataFrame({"snr": ["medium"]})

scen = pd.merge(em, snr, how="cross")

# add derived columns
scen["density"] = scen["n_emitter"] / area

scen["n_frames"] = em_min / scen["n_emitter"]
scen["n_frames"] = scen["n_frames"].apply(lambda x: int(min(max(x, frame_min), frame_max)))
scen.at[:, "phot_loc"] = scen["snr"].map({
    # "low": [1000., 1000.],
    "medium": [5000., 5000.],  # AF647, CF680
    # "high": [20000., 20000.],
})
scen["phot_scale"] = None
scen["phot_scale"] = scen["phot_loc"].apply(lambda p: [pp / 4 for pp in p])


scen.sort_values(["snr", "density"], inplace=True)
scen["index"] = np.arange(len(scen))
scen = scen.set_index(["snr", "index"])

# bg derived by other parameters
scen["bg"] = None
scen.at["medium", "bg"] = [[20., 135.]]

# dichoric
scen["choric"] = None
# scen.at["high", "choric"] = [[0.173, 0.06]]
scen.at["medium", "choric"] = [[0.173, 0.019]]
# scen.at["low", "choric"] = [[0.173, 0.06]]

# lifetimes
scen["lifetime"] = None
scen.at["medium", "lifetime"] = [[3., 1.]]

# set up target filters
scen["phot_min"] = None
# scen.loc["high", "phot_min"] = 1000.
scen.loc["medium", "phot_min"] = 100.
# scen.loc["low", "phot_min"] = 50.

scen


n_emitter   density  n_frames          phot_loc  \
snr    index                                                      
medium 0        11.624977  0.030000      1000  [5000.0, 5000.0]   
       1        15.802486  0.040781      1000  [5000.0, 5000.0]   
       2        21.481211  0.055435      1000  [5000.0, 5000.0]   
       3        29.200621  0.075357      1000  [5000.0, 5000.0]   
       4        39.694050  0.102436      1000  [5000.0, 5000.0]   
       5        53.958359  0.139248      1000  [5000.0, 5000.0]   
       6        73.348648  0.189287      1000  [5000.0, 5000.0]   
       7        99.706947  0.257309      1000  [5000.0, 5000.0]   
       8       135.537292  0.349774       737  [5000.0, 5000.0]   
       9       184.243484  0.475468       542  [5000.0, 5000.0]   
       10      250.452545  0.646330       399  [5000.0, 5000.0]   
       11      340.454254  0.878593       293  [5000.0, 5000.0]   
       12      462.798676  1.194322       216  [5000.0, 5000.0]   
       13      629.108337  1.623509       158  [5000.0, 5000.0]   
       14      855.182434  2.206927       116  [5000.0, 5000.0]   
       15     1162.497681  3.000000        86  [5000.0, 5000.0]   

                    phot_scale             bg          choric    lifetime  \
snr    index                                                                
medium 0      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       1      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       2      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       3      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       4      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       5      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       6      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       7      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       8      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       9      [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       10     [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       11     [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       12     [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       13     [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       14     [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   
       15     [1250.0, 1250.0]  [20.0, 135.0]  [0.173, 0.019]  [3.0, 1.0]   

             phot_min  
snr    index           
medium 0        100.0  
       1        100.0  
       2        100.0  
       3        100.0  
       4        100.0  
       5        100.0  
       6        100.0  
       7        100.0  
       8        100.0  
       9        100.0  
       10       100.0  
       11       100.0  
       12       100.0  
       13       100.0  
       14       100.0  
       15       100.0

In [16]:
structure = decode.simulation.structures.RandomStructure(
    xextent, yextent, zextent
)

In [17]:
xextent_psf = ((-0.5, 255.5),) * 2
yextent_psf = xextent_psf
img_shape_psf = (img_shape,) * 2

psf = decode.io.psf.load_spline(
    path_spline,
    xextent=xextent_psf,
    yextent=yextent_psf,
    img_shape=img_shape_psf,
    device=device,
    norm=psf_norm,
    kwargs_static={"max_frame_chunk": 10000, "max_roi_chunk": 10000},
)

2026-04-27 16:06:22 [info     ] Auto-loaded normalization from calibration file norm=[1.0, 5.210029736452765]


In [18]:
path = path_trafo if isinstance(path_trafo, Path) else Path(path_trafo)

offset = [-2, -23., 0.]
trafo = decode.io.trafo.load_xyz_trafo(
    path_trafo,
    scale=1 / 1000.0,
    switch_xy=True,
    shift=(1.0, 1.0, 0.0),
    reference="trafo_inv_raw",
    device=device,
)
t_mirr = decode.simulation.trafo.pos.trafo.XYZMirrorAt.from_frame_flip(
    trafo_size_ref,
    trafo_mirr_dim,
    device=device,
)
t_mirr = decode.simulation.trafo.pos.trafo.XYZChanneledTransformation(
    t_mirr,
    ch=1,
)
trafo.append(t_mirr)
trafo_shift = decode.simulation.trafo.pos.trafo.XYZShiftTransformation(
    offset, device=device
)
trafo_shift = decode.simulation.trafo.pos.trafo.XYZChanneledTransformation(trafo_shift, 1)

# final trafo
trafo.append(trafo_shift)
trafo = trafo.to("cpu")

trafo

2026-04-27 16:06:22 [info     ] Running experimental function `load_xyz_trafo` tested=True


XYZCompositTransformation([
    XYZShiftTransformation(tensor([1., 1., 0.]), cpu),
    XYZScaleTransformation(0.0010000000474974513, global),
    XYZTransformationGeneric(<function _coord_permute_smap at 0x7493ef55da20>, None, cpu, True),
    XYZTransformationMatrix(tensor([[[ 1.0000,  0.0000,  0.0000],
         [ 0.0000,  1.0000,  0.0000],
         [ 0.0000,  0.0000,  1.0000]],

        [[ 1.0023, -0.0352,  0.0066],
         [-0.0380, -1.0035, -0.0027],
         [ 0.0277,  0.5205,  1.0000]]])),
    XYZTransformationGeneric(<function _coord_permute_smap at 0x7493ef55da20>, None, cpu, True),
    XYZScaleTransformation(1000.0, global),
    XYZShiftTransformation(tensor([-1., -1., -0.]), cpu),
    XYZChanneledTransformation(XYZMirrorAt(axis=0, at=511, device=cpu), 1),
    XYZChanneledTransformation(XYZShiftTransformation(tensor([ -2., -23.,   0.]), cpu), 1)
])

In [19]:
cam = decode.simulation.camera.CameraPerfect(sensor_size=img_shape, device=device)
cam = [cam] * 2

choric = decode.simulation.trafo.photon.trafo.MultiChoricSplitter(choric_mat)

mic = decode.simulation.microscope.MicroscopeMultiChannel(
    psf=psf,
    noise=cam, # manually, because then we can batch
    frame_range=(0, n_frames),
    ch_range=(0, 2),
)

# Sample

In [20]:
# independents
scen_x = scen.copy()

scen_x["em"] = None
scen_x["em_ch"] = None
scen_x["bg_frame"] = None

torch.manual_seed(random_seed)

for ix, s in scen_x.iterrows():
    flux = (s["phot_loc"], s["phot_scale"])

    em = []
    # loop over codes
    for i, (loc, scale, lifetime) in enumerate(zip(s["phot_loc"], s["phot_scale"], s["lifetime"], strict=True)):
        e = decode.simulation.sampler.EmitterSamplerBlinking(
                structure=structure,
                flux=(loc, scale),
                em_num=s["n_emitter"],
                lifetime=lifetime,
                frame_range=(0, s["n_frames"]),
                code=None,
                n_channels=1,
                xy_unit="px",
                px_size=px_size,
            ).sample()
        e.code = torch.ones(len(e), dtype=torch.long) * i
        em.append(e)
    em = decode.EmitterSet.cat(em)

    # filter out low photon ones
    em = em[em.phot.squeeze() >= s["phot_min"]]

    choric_lin = torch.as_tensor(s["choric"])
    choric_mat = decode.simulation.trafo.photon.utils.dual_from_ratio(choric_lin, 1)
    choric = decode.simulation.trafo.photon.trafo.MultiChoricSplitter(choric_mat)

    e_ch = em.clone()
    e_ch.phot = choric.forward(e_ch.phot.squeeze(), e_ch.code)
    e_ch.xyz = trafo.forward(e_ch.xyz)

    scen_x.at[ix, "em"] = em
    scen_x.at[ix, "em_ch"] = e_ch
    scen_x.at[ix, "bg_frame"] = [torch.ones(img_shape) * bg for bg in s["bg"]]


In [21]:
scen_y = pd.DataFrame(index=scen_x.index, columns=["frame"])

for ix, s in scen_x.iterrows():
    e_ch = s["em_ch"]
    bg_frame = s["bg_frame"]

    frames = mic.forward(e_ch, bg_frame, ix_low=0, ix_high=s["n_frames"])
    
    scen_y.at[ix, "frame"] = [ff.cpu() for ff in frames]

2026-04-27 16:06:36 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:06:38 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:06:40 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:06:41 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:06:44 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:06:46 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:06:50 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:06:54 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:07:00 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:07:04 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:07:07 [warning  ] Overwriting/Inferring code from phot instead of em.code.
2026-04-27 16:07:09 [

In [22]:
em = scen_x["em"].to_list()
em_ch = scen_x["em_ch"].to_list()
bg = scen_x["bg_frame"].to_list()

frames = scen_y["frame"].to_list()

# Export

In [ ]:
# make sure not to overwrite
path_save = path_out / f"n_frames-{n_frames}_n_steps-{n_steps}_seed-{random_seed}-3"
# path_save.mkdir(exist_ok=False)

print(path_save)

../../data/density/dual_color/n_frames-1000_n_steps-16_seed-42-3


In [26]:
# export scenarios
path_scen = path_save / "scenarios.pq"
scen.to_parquet(path_scen)

# export ground truth
gt = {
    "em": em,
    "em_ch": em_ch,
    "bg": bg,
}
path_gt = path_save / "gt.pt"
torch.save(gt, path_gt)

# export frames
path_frames = path_save / "frames.pt"
torch.save(frames, path_frames)

In [27]:
# export frames as tiffs per SNR, for this accumulate the frames per scenario and bookkeep the lower and upper index
scen_bulk = pd.DataFrame(index=scen.index)

# make sure to not have negative values
offset = 100

# bookkeep frame length of each scenario and infer the lower and upper index
frame_len = [len(f[0]) for f in frames]
scen_bulk["frame_len"] = frame_len
scen_bulk["ix_low"] = None


# per snr, infer the lower and upper index
aux_cumsum = scen_bulk.groupby("snr").apply(lambda x: x["frame_len"].cumsum())
if len(scen_bulk.index.get_level_values(0).unique()) == 1:
    aux_cumsum = aux_cumsum.T
else:
    aux_cumsum = aux_cumsum.droplevel(0)
scen_bulk["ix_high"] = aux_cumsum

# infer ix_low
scen_bulk["ix_low"] = scen_bulk["ix_high"].shift(1)
# per snr, fill the first ix_low with 0
ix = scen_bulk.groupby(level=0).head(1).index
scen_bulk.loc[ix, "ix_low"] = 0
scen_bulk["ix_low"] = scen_bulk["ix_low"].astype(int)

#infer path
scen_bulk["name"] = None
for s in scen_bulk.index.get_level_values(0).unique():
    scen_bulk.loc[s, "name"] = f"frames_snr-{s}_offset-{offset}_ome.tif"

# export scenarios for saving
path_scen_bulk = path_save / "scen_bulk.pq"
scen_bulk.to_parquet(path_scen_bulk)

scen_bulk

frame_len  ix_low  ix_high                                  name
snr    index                                                                  
medium 0           1000       0     1000  frames_snr-medium_offset-100_ome.tif
       1           1000    1000     2000  frames_snr-medium_offset-100_ome.tif
       2           1000    2000     3000  frames_snr-medium_offset-100_ome.tif
       3           1000    3000     4000  frames_snr-medium_offset-100_ome.tif
       4           1000    4000     5000  frames_snr-medium_offset-100_ome.tif
       5           1000    5000     6000  frames_snr-medium_offset-100_ome.tif
       6           1000    6000     7000  frames_snr-medium_offset-100_ome.tif
       7           1000    7000     8000  frames_snr-medium_offset-100_ome.tif
       8            737    8000     8737  frames_snr-medium_offset-100_ome.tif
       9            542    8737     9279  frames_snr-medium_offset-100_ome.tif
       10           399    9279     9678  frames_snr-medium_offset-100_ome.tif
       11           293    9678     9971  frames_snr-medium_offset-100_ome.tif
       12           216    9971    10187  frames_snr-medium_offset-100_ome.tif
       13           158   10187    10345  frames_snr-medium_offset-100_ome.tif
       14           116   10345    10461  frames_snr-medium_offset-100_ome.tif
       15            86   10461    10547  frames_snr-medium_offset-100_ome.tif

In [28]:

frames_cat_flip = [torch.cat([f[0], torch.flip(f[1], dims=[1])], dim=1) for f in frames] 
frames_cat = [torch.cat(f, dim=1) for f in frames]  # put channels next to each other
frames_cat[0].size()

torch.Size([1000, 512, 256])

In [ ]:
# export all scenarios as tiff's (combined, such that we can fit faster)
# per frame, accumulate the frames and save

scen_save = scen_bulk.reset_index().set_index("name").sort_index()

for name in scen_save.index.unique():
    sub = scen_save.loc[name]

    # collect frames in sub_scenario
    f = torch.cat([frames_cat[i] for i in sub["index"].values], dim=0)
    f_flip = torch.cat([frames_cat_flip[i] for i in sub["index"].values], dim=0)
    f = f + offset
    f_flip = f_flip + offset
    path_tiff = path_save / name
    path_tiff_flip = path_save / name.replace(".tif", "_flip.tif")
    tifffile.imwrite(path_tiff, f.long().numpy().astype(np.int32), ome=True)
    # tifffile.imwrite(path_tiff_flip, f_flip.long().numpy().astype(np.int32), ome=True) # for smap test